# P98 — RRT-Connect: un enfoque eficiente para planificación de caminos de consulta única

## 1. Título y paper

**Paper:** *RRT-Connect: An Efficient Approach to Single-Query Path Planning*  
**Autoría:** James J. Kuffner, Steven M. LaValle  
**Año y venue:** 2000 · Proceedings of ICRA 2000, 995–1001  
**Nivel:** L3 · **Motor:** `rrt`  
**Ficha completa:** [`P98_rrt`](../../papers/foundational/P98_rrt/README.md)

**Hito:** Planifica en espacios continuos de muchas dimensiones sin discretizarlos, creciendo un árbol hacia muestras aleatorias.

- [doi:10.1109/ROBOT.2000.844730](https://doi.org/10.1109/ROBOT.2000.844730)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Un brazo de siete articulaciones tiene un espacio de configuración de siete dimensiones. Discretizarlo para aplicar búsqueda en grafo produce un número de celdas astronómico, y los métodos de campos potenciales se quedan atrapados en mínimos locales.
2. Ejecutar una implementación mínima de la propuesta: Muestrear configuraciones al azar y extender el árbol desde el nodo más cercano hacia cada muestra. El árbol se sesga solo hacia las regiones no exploradas, y con dos árboles que crecen uno hacia el otro la convergencia es mucho más rápida.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P67
- Kavraki et al. (1996), mapas de caminos probabilísticos


## 4. Intuición

Un brazo de siete articulaciones tiene un espacio de configuración de siete dimensiones. Discretizarlo es imposible: con diez pasos por eje son diez millones de celdas. RRT no lo discretiza — lanza muestras al azar y crece un árbol hacia ellas.


## 5. Concepto mínimo

```text
repetir:
    q_rand ← muestra aleatoria del espacio (a veces, la meta)
    q_near ← nodo del árbol más cercano a q_rand
    q_new  ← avanzar un paso desde q_near hacia q_rand
    si el segmento está libre: añadir q_new al árbol

El árbol se sesga SOLO hacia las regiones grandes no exploradas (Voronoi).
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('rrt', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuántos nodos hacen falta frente a las celdas de una rejilla equivalente?
2. ¿Es óptimo el camino que devuelve?
3. ¿Qué aporta sesgar el muestreo hacia la meta?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('rrt', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('rrt', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

El árbol encuentra un camino con **297 nodos** sobre un espacio continuo; una rejilla de 2×2 unidades tendría 2 500 celdas. El camino mide 264 frente a los 127,3 de la línea recta: un **107 % de exceso**. RRT es probabilísticamente completo, **no óptimo**. Y el sesgo importa: sin él, 394 nodos; con un 20 %, 191.


## 10. Comentario pedagógico

Ese exceso del 107 % no es un defecto de implementación: es lo que hace RRT. Encuentra *un* camino rápido, con aspecto de zigzag, y en la práctica se suaviza después. RRT* (2011) añade optimalidad asintótica a costa de más cómputo, y esa es exactamente la misma disyuntiva que planteaba [A*](../../papers/foundational/P67_a_estrella/README.md) entre voraz y óptimo.


## 11. Error o anti-patrón deliberado

Anti-patrón: usar el camino de RRT tal cual sale.


In [ ]:
print('El camino de RRT tiene zigzags y nodos inutiles: no esta pensado para ejecutarse asi.')
print('En la practica se poda (quitar nodos intermedios visibles) y se suaviza.')
print('Saltarse ese paso produce trayectorias que el robot no puede seguir bien.')

## 12. Corrección

Lo que RRT garantiza y lo que no:


In [ ]:
r = run_paper_lab('rrt', seed=7)['result']
print('nodos expandidos      :', r['con_sesgo_5_por_ciento']['nodos_expandidos'])
print('celdas de una rejilla :', r['celdas_de_una_rejilla_equivalente'])
print('longitud del camino   :', r['con_sesgo_5_por_ciento']['longitud'],
      'vs recta', r['distancia_en_linea_recta'])
print('es optimo             :', r['es_optimo'])

## 13. Desafío guiado

Compara los nodos necesarios con sesgo 0 %, 5 % y 20 %, y explica por qué demasiado sesgo también sería malo.


In [ ]:
r = run_paper_lab('rrt', seed=3)['result']
show(r)

## 14. Desafío autónomo

Implementa RRT sobre un espacio con un pasillo estrecho entre dos salas. Mide cuántos nodos hacen falta y compáralo con un espacio abierto: ahí se ve el punto débil del muestreo uniforme.


## 15. Evidencia de aprendizaje

Guarda la comparación de nodos frente a rejilla y tu explicación de completitud probabilística frente a optimalidad.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P98_rrt/README.md) · evaluación formal: [`assessments/papers/P98_rrt.md`](../../assessments/papers/P98_rrt.md)


## 16. Cierre

Ya se puede planificar un camino. Pero todo esto supone saber dónde está el robot, y esa suposición es justamente el problema.


## 17. Conexión con el siguiente hito



Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
